# 🎭 Self-Supervised Image Representation Learning using Masked Autoencoders (MAE)

## Assignment Implementation

This notebook implements a complete **Masked Autoencoder (MAE)** system for self-supervised visual representation learning.

### Architecture Overview
- **Encoder**: ViT-Base (B/16) - 768 dim, 12 layers, 12 heads (~86M parameters)
- **Decoder**: ViT-Small (S/16) - 384 dim, 12 layers, 6 heads (~22M parameters)
- **Masking Ratio**: 75% (only 25% visible patches)

### Key Features
- Asymmetric encoder-decoder design
- Mixed precision training (AMP)
- Multi-GPU support (DataParallel)
- Cosine learning rate scheduler with warmup
- PSNR and SSIM evaluation metrics
- Gradio deployment app

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install einops gradio -q

# Display GPU info
!nvidia-smi

In [ ]:
# Import libraries
import os
import sys
import time
import json
import math
import random
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR

from torchvision import transforms
from PIL import Image

import matplotlib.pyplot as plt
from einops import rearrange, repeat

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# Configuration
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## 🚀 Optimized Training Strategy (3-Hour Target + HuggingFace Ready)

### Key Optimizations:
1. **Same Architecture**: ViT-Base encoder (768d, 12 layers) + ViT-Small decoder (384d, 12 layers)
2. **Data Subset**: Use 20% of training data for ~3 hour training time
3. **Increased Epochs**: 25 epochs with reduced data
4. **Pickle Serialization**: Efficient model saving with gzip compression
5. **HuggingFace Export**: Optimized model-only format (~250 MB)

### Time Optimization:
- **Target**: Complete 25 epochs in ~3 hours on Kaggle T4 GPU
- **Strategy**: 20% data → ~312 batches/epoch → ~7 min/epoch
- **Total**: 25 × 7 min = ~175 minutes (2.9 hours) ✅

### File Size Optimization:
- **PyTorch checkpoint**: ~850 MB (full state)
- **Pickle compressed**: ~400 MB (full state)
- **HuggingFace ready**: ~250 MB (model only, max compression)
- **Result**: Easy upload to HuggingFace (< 500 MB limit) ✅

**Note:** Model parameters remain exactly as specified (~86M encoder + ~22M decoder)

In [ ]:
import pickle
import gzip
from pathlib import Path

# 🎯 OPTIMIZED CONFIG - Same Architecture, Optimized for 3-hour training + HuggingFace upload
CONFIG_OPTIMIZED = {
    'img_size': 224,
    'patch_size': 16,
    'mask_ratio': 0.75,
    
    # Encoder (ViT-Base) - SAME as original assignment specification
    'encoder_embed_dim': 768,      # ViT-Base: 768d
    'encoder_depth': 12,           # ViT-Base: 12 layers
    'encoder_num_heads': 12,       # ViT-Base: 12 heads
    
    # Decoder (ViT-Small) - SAME as original assignment specification
    'decoder_embed_dim': 384,      # ViT-Small: 384d
    'decoder_depth': 12,           # ViT-Small: 12 layers
    'decoder_num_heads': 6,        # ViT-Small: 6 heads
    
    # Training - Optimized for ~3 hours on Kaggle T4 GPU
    'batch_size': 64,
    'learning_rate': 1.5e-4,
    'weight_decay': 0.05,
    'warmup_epochs': 2,
    'total_epochs': 25,            # 25 epochs in ~3 hours
    'num_workers': 4,
    'gradient_clip': 1.0,
    
    # Data optimization - Tuned for 3-hour training window
    'use_data_subset': True,
    'data_subset_ratio': 0.20,     # 20% of data → ~7 min/epoch → ~3 hours total
}

print("✅ Optimized Training Configuration:")
print(f"   Architecture: ViT-Base (B/16) encoder + ViT-Small (S/16) decoder")
print(f"   Encoder: {CONFIG_OPTIMIZED['encoder_embed_dim']}d × {CONFIG_OPTIMIZED['encoder_depth']} layers × {CONFIG_OPTIMIZED['encoder_num_heads']} heads (~86M params)")
print(f"   Decoder: {CONFIG_OPTIMIZED['decoder_embed_dim']}d × {CONFIG_OPTIMIZED['decoder_depth']} layers × {CONFIG_OPTIMIZED['decoder_num_heads']} heads (~22M params)")
print(f"   Total Parameters: ~108M")
print(f"")
print(f"   ⏱️  Training Time Optimization:")
print(f"   Epochs: {CONFIG_OPTIMIZED['total_epochs']}")
print(f"   Data: {int(CONFIG_OPTIMIZED['data_subset_ratio']*100)}% of dataset")
print(f"   Estimated time: ~7 min/epoch × 25 epochs = ~3 hours on T4 GPU")
print(f"")
print(f"   💾 HuggingFace Upload Ready:")
print(f"   Save Format: Pickle with gzip compression")
print(f"   Expected size: ~400-450 MB (suitable for HF upload)")
print(f"   Model only (no optimizer): ~430 MB → ~200-250 MB compressed")

In [ ]:
# 📦 Model Saving Utilities - Optimized for HuggingFace Spaces

def save_model_pickle(model, filepath, compress=True, config=None):
    """Save model using pickle with optional gzip compression."""
    if isinstance(model, nn.DataParallel):
        model_state = model.module.state_dict()
    else:
        model_state = model.state_dict()
    
    save_data = {
        'model_state_dict': model_state,
        'config': config
    }
    
    if compress:
        with gzip.open(filepath, 'wb') as f:
            pickle.dump(save_data, f, protocol=pickle.HIGHEST_PROTOCOL)
    else:
        with open(filepath, 'wb') as f:
            pickle.dump(save_data, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    file_size = os.path.getsize(filepath) / (1024 ** 2)
    print(f"   Saved model to: {filepath}")
    print(f"   File size: {file_size:.2f} MB")
    return file_size


def load_model_pickle(filepath, model, compress=True):
    """Load model from pickle file."""
    if compress:
        with gzip.open(filepath, 'rb') as f:
            save_data = pickle.load(f)
    else:
        with open(filepath, 'rb') as f:
            save_data = pickle.load(f)
    
    model.load_state_dict(save_data['model_state_dict'])
    return save_data.get('config', None)


def save_for_huggingface_torch(model, save_dir, config):
    """
    Save model in PyTorch format - BEST for HuggingFace Spaces.
    Saves ONLY model weights (no optimizer, scheduler, history).
    Smallest file size with maximum compatibility.
    
    Args:
        model: Trained model
        save_dir: Directory to save files
        config: Model configuration
    
    Returns:
        Path and file size in MB
    """
    if isinstance(model, nn.DataParallel):
        model_to_save = model.module
    else:
        model_to_save = model
    
    # Save in PyTorch format - Model weights + minimal config
    hf_path = os.path.join(save_dir, 'mae_model_weights.pth')
    
    # Only save what's needed: model weights and architecture config
    save_dict = {
        'model_state_dict': model_to_save.state_dict(),
        'config': {
            'img_size': config['img_size'],
            'patch_size': config['patch_size'],
            'encoder_embed_dim': config['encoder_embed_dim'],
            'encoder_depth': config['encoder_depth'],
            'encoder_num_heads': config['encoder_num_heads'],
            'decoder_embed_dim': config['decoder_embed_dim'],
            'decoder_depth': config['decoder_depth'],
            'decoder_num_heads': config['decoder_num_heads'],
            'mask_ratio': config['mask_ratio']
        }
    }
    
    # Save with PyTorch - automatically optimized
    torch.save(save_dict, hf_path, _use_new_zipfile_serialization=True)
    
    file_size = os.path.getsize(hf_path) / (1024 ** 2)
    
    print(f"\n🤗 HuggingFace Spaces Ready - PyTorch Format:")
    print(f"   File: mae_model_weights.pth")
    print(f"   Path: {hf_path}")
    print(f"   Size: {file_size:.2f} MB")
    print(f"   Format: PyTorch (.pth) - weights only")
    print(f"   Status: {'✅ Perfect for HF Spaces (< 500MB)' if file_size < 500 else '⚠️  Large file'}")
    print(f"   Upload: ✅ Can be directly uploaded and loaded in Gradio app")
    
    return hf_path, file_size


def load_model_from_hf_torch(filepath):
    """Load model weights from HuggingFace PyTorch format."""
    checkpoint = torch.load(filepath, map_location='cpu')
    return checkpoint['model_state_dict'], checkpoint['config']


print("✅ Model saving utilities defined")
print("   Formats supported:")
print("   - PyTorch .pth (RECOMMENDED for HuggingFace Spaces)")
print("   - Pickle with gzip (alternative)")
print("   ")
print("   📌 For HuggingFace Spaces: Use .pth format")
print("      - Smaller file size (~430 MB)")
print("      - Faster loading in Gradio")
print("      - Native PyTorch compatibility")
print("      - Easy integration with Spaces")

## 📋 Configuration Summary

The optimized configuration maintains the **exact architecture** specified in the assignment:

### Encoder (ViT-Base B/16)
- Hidden Dimension: **768**
- Layers: **12**
- Attention Heads: **12**
- Parameters: **~86M**

### Decoder (ViT-Small S/16)
- Hidden Dimension: **384**
- Layers: **12**
- Attention Heads: **6**
- Parameters: **~22M**

### Total Model
- **~108M parameters** (exact match with requirements)
- Architecture verification will be performed after model definition below

In [ ]:
# Paths - Auto-detect Kaggle dataset path
import glob

def explore_directory(path, depth=0, max_depth=4):
    """Recursively explore directory structure."""
    if depth > max_depth or not os.path.exists(path):
        return
    
    indent = "  " * depth
    try:
        items = os.listdir(path)
        for item in items[:15]:  # Limit items shown
            item_path = os.path.join(path, item)
            if os.path.isdir(item_path):
                print(f"{indent}📁 {item}/")
                explore_directory(item_path, depth + 1, max_depth)
            else:
                print(f"{indent}📄 {item}")
        if len(items) > 15:
            print(f"{indent}... and {len(items) - 15} more items")
    except PermissionError:
        print(f"{indent}[Permission Denied]")

def find_tiny_imagenet_dir():
    """Auto-detect TinyImageNet dataset directory by searching for train folder."""
    
    # Search patterns - look for directories containing both 'train' and 'val'
    base_paths = [
        '/kaggle/input',
    ]
    
    for base in base_paths:
        if not os.path.exists(base):
            continue
            
        # Walk through all subdirectories
        for root, dirs, files in os.walk(base):
            # Check if this directory has 'train' and 'val' subdirectories
            if 'train' in dirs and 'val' in dirs:
                train_path = os.path.join(root, 'train')
                # Verify train has class subdirectories (not just files)
                try:
                    train_contents = os.listdir(train_path)
                    if len(train_contents) > 10:  # TinyImageNet has 200 classes
                        # Check if subdirs have images
                        sample_class = os.path.join(train_path, train_contents[0])
                        if os.path.isdir(sample_class):
                            print(f"✓ Found TinyImageNet at: {root}")
                            return root
                except:
                    continue
    
    return None


# Try to auto-detect
detected_path = find_tiny_imagenet_dir()

if detected_path:
    DATA_DIR = detected_path
else:
    # Fallback to common paths
    potential_paths = [
        '/kaggle/input/tiny-imagenet-200-python/tiny-imagenet-200',
        '/kaggle/input/tiny-imagenet/tiny-imagenet-200',
        '/kaggle/input/tiny-imagenet-200',
        '/kaggle/input/tiny-imagenet200/tiny-imagenet-200',
        'data/tiny-imagenet-200',
        './tiny-imagenet-200',
    ]
    
    DATA_DIR = None
    for path in potential_paths:
        if os.path.exists(path):
            DATA_DIR = path
            break

# Verify and show structure
if DATA_DIR:
    print(f"\n✓ Using DATA_DIR: {DATA_DIR}")
    print("\nDirectory structure:")
    explore_directory(DATA_DIR, max_depth=2)
else:
    print("❌ Could not find TinyImageNet directory!")
    print("\nSearching in /kaggle/input:")
    explore_directory('/kaggle/input', max_depth=3)

TRAIN_DIR = os.path.join(DATA_DIR, 'train') if DATA_DIR else None
VAL_DIR = os.path.join(DATA_DIR, 'val') if DATA_DIR else None
SAVE_DIR = '/kaggle/working/mae_checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

if DATA_DIR and os.path.exists(DATA_DIR):
    print(f"\n✓ DATA_DIR: {DATA_DIR}")
    print(f"✓ TRAIN_DIR: {TRAIN_DIR}")
    print(f"✓ VAL_DIR: {VAL_DIR}")
    print(f"✓ SAVE_DIR: {SAVE_DIR}")
else:
    print(f"❌ DATA_DIR does not exist!")

# Training configuration
CONFIG = {
    'img_size': 224,
    'patch_size': 16,
    'mask_ratio': 0.75,
    
    # Encoder (ViT-Base)
    'encoder_embed_dim': 768,
    'encoder_depth': 12,
    'encoder_num_heads': 12,
    
    # Decoder (ViT-Small)
    'decoder_embed_dim': 384,
    'decoder_depth': 12,
    'decoder_num_heads': 6,
    
    # Training
    'batch_size': 64,
    'learning_rate': 1.5e-4,
    'weight_decay': 0.05,
    'warmup_epochs': 1,
    'total_epochs': 10,
    'num_workers': 4,
    'gradient_clip': 1.0,
}

print("\nConfiguration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 2. Model Architecture

### 2.1 Positional Embeddings

In [ ]:
def get_2d_sincos_pos_embed(embed_dim, grid_size):
    """
    Generate 2D sinusoidal positional embeddings.
    """
    grid_h = np.arange(grid_size, dtype=np.float32)
    grid_w = np.arange(grid_size, dtype=np.float32)
    grid = np.meshgrid(grid_w, grid_h)
    grid = np.stack(grid, axis=0)
    grid = grid.reshape([2, 1, grid_size, grid_size])
    
    pos_embed = get_2d_sincos_pos_embed_from_grid(embed_dim, grid)
    return pos_embed


def get_2d_sincos_pos_embed_from_grid(embed_dim, grid):
    assert embed_dim % 2 == 0
    emb_h = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[0])
    emb_w = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[1])
    emb = np.concatenate([emb_h, emb_w], axis=1)
    return emb


def get_1d_sincos_pos_embed_from_grid(embed_dim, pos):
    assert embed_dim % 2 == 0
    omega = np.arange(embed_dim // 2, dtype=np.float32)
    omega /= embed_dim / 2.
    omega = 1. / 10000**omega
    
    pos = pos.reshape(-1)
    out = np.einsum('m,d->md', pos, omega)
    
    emb_sin = np.sin(out)
    emb_cos = np.cos(out)
    emb = np.concatenate([emb_sin, emb_cos], axis=1)
    return emb

print("Positional embedding functions defined.")

### 2.2 Transformer Building Blocks

In [ ]:
class PatchEmbedding(nn.Module):
    """Convert image to patches and embed them."""
    
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        
    def forward(self, x):
        x = self.proj(x)
        x = rearrange(x, 'b e h w -> b (h w) e')
        return x


class MultiHeadSelfAttention(nn.Module):
    """Multi-Head Self Attention block."""
    
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.attn_drop = nn.Dropout(dropout)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.proj_drop = nn.Dropout(dropout)
        
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class MLP(nn.Module):
    """MLP block with GELU activation."""
    
    def __init__(self, embed_dim, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        hidden_dim = int(embed_dim * mlp_ratio)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x


class TransformerBlock(nn.Module):
    """Transformer encoder block."""
    
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim, mlp_ratio, dropout)
        
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

print("Transformer building blocks defined.")

### 2.3 ViT Encoder (ViT-Base)

In [ ]:
class ViTEncoder(nn.Module):
    """
    Vision Transformer Encoder (ViT-Base)
    - Patch Size: 16x16
    - Hidden Dimension: 768
    - Transformer Layers: 12
    - Attention Heads: 12
    - Parameters: ~86M
    """
    
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_channels=3,
        embed_dim=768,
        depth=12,
        num_heads=12,
        mlp_ratio=4.0,
        dropout=0.0
    ):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        self.num_patches = self.patch_embed.num_patches
        
        # Positional embeddings (learnable)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        self.pos_drop = nn.Dropout(dropout)
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
        
        # Initialize positional embeddings
        self._init_weights()
        
    def _init_weights(self):
        pos_embed = get_2d_sincos_pos_embed(
            self.pos_embed.shape[-1],
            int(self.num_patches ** 0.5)
        )
        self.pos_embed.data.copy_(torch.from_numpy(pos_embed).float().unsqueeze(0))
        
    def forward(self, x, mask_indices=None):
        """
        Forward pass for encoder.
        Only processes visible patches (NOT mask tokens).
        """
        x = self.patch_embed(x)
        x = x + self.pos_embed
        
        # Keep only visible patches
        if mask_indices is not None:
            B, N, D = x.shape
            mask_indices_expanded = mask_indices.unsqueeze(-1).expand(-1, -1, D)
            x = torch.gather(x, dim=1, index=mask_indices_expanded)
        
        x = self.pos_drop(x)
        
        for block in self.blocks:
            x = block(x)
            
        x = self.norm(x)
        return x

print("ViT Encoder defined.")

### 2.4 ViT Decoder (ViT-Small)

In [ ]:
class ViTDecoder(nn.Module):
    """
    Vision Transformer Decoder (ViT-Small)
    - Patch Size: 16x16
    - Hidden Dimension: 384
    - Transformer Layers: 12
    - Attention Heads: 6
    - Parameters: ~22M
    """
    
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        embed_dim=384,
        depth=12,
        num_heads=6,
        mlp_ratio=4.0,
        dropout=0.0,
        encoder_embed_dim=768
    ):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        
        # Project encoder output to decoder dimension
        self.encoder_to_decoder = nn.Linear(encoder_embed_dim, embed_dim)
        
        # Learnable mask token
        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        # Positional embeddings for decoder
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
        
        # Prediction head: project to pixel space
        self.pred = nn.Linear(embed_dim, patch_size ** 2 * 3)
        
        self._init_weights()
        
    def _init_weights(self):
        pos_embed = get_2d_sincos_pos_embed(
            self.pos_embed.shape[-1],
            int(self.num_patches ** 0.5)
        )
        self.pos_embed.data.copy_(torch.from_numpy(pos_embed).float().unsqueeze(0))
        nn.init.normal_(self.mask_token, std=0.02)
        
    def forward(self, x, visible_indices, mask_indices):
        """
        Forward pass for decoder.
        Combines encoder output with learnable mask tokens.
        """
        B, num_visible, _ = x.shape
        num_masked = mask_indices.shape[1]
        
        # Project encoder output to decoder dimension
        x = self.encoder_to_decoder(x)
        
        # Create mask tokens - ensure same dtype as x for mixed precision compatibility
        mask_tokens = self.mask_token.expand(B, num_masked, -1).to(dtype=x.dtype)
        
        # Combine visible tokens and mask tokens
        # Use same dtype as x for mixed precision compatibility
        full_tokens = torch.zeros(
            B, self.num_patches, x.shape[-1],
            device=x.device, dtype=x.dtype
        )
        
        visible_indices_expanded = visible_indices.unsqueeze(-1).expand(-1, -1, x.shape[-1])
        full_tokens.scatter_(1, visible_indices_expanded, x)
        
        mask_indices_expanded = mask_indices.unsqueeze(-1).expand(-1, -1, x.shape[-1])
        full_tokens.scatter_(1, mask_indices_expanded, mask_tokens)
        
        # Add positional embeddings (cast to same dtype for mixed precision)
        full_tokens = full_tokens + self.pos_embed.to(dtype=x.dtype)
        
        for block in self.blocks:
            full_tokens = block(full_tokens)
            
        full_tokens = self.norm(full_tokens)
        pred = self.pred(full_tokens)
        
        return pred

print("ViT Decoder defined.")

### 2.5 Complete MAE Model

In [ ]:
class MaskedAutoencoder(nn.Module):
    """
    Masked Autoencoder for Self-Supervised Learning
    Asymmetric encoder-decoder architecture.
    """
    
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_channels=3,
        encoder_embed_dim=768,
        encoder_depth=12,
        encoder_num_heads=12,
        decoder_embed_dim=384,
        decoder_depth=12,
        decoder_num_heads=6,
        mlp_ratio=4.0,
        mask_ratio=0.75,
        dropout=0.0
    ):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.mask_ratio = mask_ratio
        
        self.encoder = ViTEncoder(
            img_size=img_size,
            patch_size=patch_size,
            in_channels=in_channels,
            embed_dim=encoder_embed_dim,
            depth=encoder_depth,
            num_heads=encoder_num_heads,
            mlp_ratio=mlp_ratio,
            dropout=dropout
        )
        
        self.decoder = ViTDecoder(
            img_size=img_size,
            patch_size=patch_size,
            embed_dim=decoder_embed_dim,
            depth=decoder_depth,
            num_heads=decoder_num_heads,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
            encoder_embed_dim=encoder_embed_dim
        )
        
    def patchify(self, imgs):
        """Convert images to patches."""
        p = self.patch_size
        x = rearrange(imgs, 'b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=p, p2=p)
        return x
    
    def unpatchify(self, x):
        """Convert patches back to images."""
        p = self.patch_size
        h = w = self.img_size // p
        x = rearrange(x, 'b (h w) (p1 p2 c) -> b c (h p1) (w p2)', h=h, w=w, p1=p, p2=p, c=3)
        return x
    
    def random_masking(self, batch_size, device, mask_ratio=None):
        """Generate random mask indices."""
        if mask_ratio is None:
            mask_ratio = self.mask_ratio
            
        num_patches = self.num_patches
        num_visible = int(num_patches * (1 - mask_ratio))
        
        noise = torch.rand(batch_size, num_patches, device=device)
        ids_shuffle = torch.argsort(noise, dim=1)
        
        visible_indices = ids_shuffle[:, :num_visible]
        mask_indices = ids_shuffle[:, num_visible:]
        
        visible_indices = torch.sort(visible_indices, dim=1)[0]
        mask_indices = torch.sort(mask_indices, dim=1)[0]
        
        return visible_indices, mask_indices
    
    def forward(self, imgs, mask_ratio=None):
        """Forward pass."""
        B = imgs.shape[0]
        device = imgs.device
        
        visible_indices, mask_indices = self.random_masking(B, device, mask_ratio)
        latent = self.encoder(imgs, visible_indices)
        pred = self.decoder(latent, visible_indices, mask_indices)
        target = self.patchify(imgs)
        
        return pred, target, mask_indices
    
    def forward_loss(self, imgs, mask_ratio=None):
        """Compute forward pass and loss (MSE on masked patches only)."""
        pred, target, mask_indices = self.forward(imgs, mask_ratio)
        
        B = imgs.shape[0]
        mask_indices_expanded = mask_indices.unsqueeze(-1).expand(-1, -1, pred.shape[-1])
        
        pred_masked = torch.gather(pred, dim=1, index=mask_indices_expanded)
        target_masked = torch.gather(target, dim=1, index=mask_indices_expanded)
        
        loss = F.mse_loss(pred_masked, target_masked)
        
        return loss, pred, target, mask_indices


def create_mae_model(config):
    """Factory function to create MAE model from config."""
    model = MaskedAutoencoder(
        img_size=config['img_size'],
        patch_size=config['patch_size'],
        encoder_embed_dim=config['encoder_embed_dim'],
        encoder_depth=config['encoder_depth'],
        encoder_num_heads=config['encoder_num_heads'],
        decoder_embed_dim=config['decoder_embed_dim'],
        decoder_depth=config['decoder_depth'],
        decoder_num_heads=config['decoder_num_heads'],
        mask_ratio=config['mask_ratio']
    )
    return model


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Test the model
model = create_mae_model(CONFIG)
print(f"\nModel created successfully!")
print(f"Total parameters: {count_parameters(model):,}")
print(f"Encoder parameters: {count_parameters(model.encoder):,}")
print(f"Decoder parameters: {count_parameters(model.decoder):,}")

# Test forward pass
x = torch.randn(2, 3, 224, 224)
loss, pred, target, mask_indices = model.forward_loss(x)
print(f"\nTest forward pass:")
print(f"  Input shape: {x.shape}")
print(f"  Prediction shape: {pred.shape}")
print(f"  Target shape: {target.shape}")
print(f"  Mask indices shape: {mask_indices.shape}")
print(f"  Loss: {loss.item():.4f}")

## 3. Dataset: TinyImageNet

In [ ]:
# 🔍 Verify Optimized Configuration Matches Requirements

print("\n" + "="*70)
print("📋 OPTIMIZED CONFIGURATION VERIFICATION")
print("="*70)

# Create model with optimized config
model_opt_test = create_mae_model(CONFIG_OPTIMIZED)

print(f"\n✅ ENCODER (ViT-Base B/16):")
print(f"   Patch Size:      16×16")
print(f"   Image Size:      224×224")
print(f"   Hidden Dim:      {CONFIG_OPTIMIZED['encoder_embed_dim']}")
print(f"   Layers:          {CONFIG_OPTIMIZED['encoder_depth']}")
print(f"   Attention Heads: {CONFIG_OPTIMIZED['encoder_num_heads']}")
print(f"   Parameters:      {count_parameters(model_opt_test.encoder):,} (~86M)")

print(f"\n✅ DECODER (ViT-Small S/16):")
print(f"   Patch Size:      16×16")
print(f"   Hidden Dim:      {CONFIG_OPTIMIZED['decoder_embed_dim']}")
print(f"   Layers:          {CONFIG_OPTIMIZED['decoder_depth']}")
print(f"   Attention Heads: {CONFIG_OPTIMIZED['decoder_num_heads']}")
print(f"   Parameters:      {count_parameters(model_opt_test.decoder):,} (~22M)")

print(f"\n✅ TOTAL MODEL:")
print(f"   Total Parameters: {count_parameters(model_opt_test):,}")
print(f"   Matches original: {count_parameters(model_opt_test) == count_parameters(model)}")
print(f"   Expected:         ~108M parameters")

# Test forward pass with optimized config
x_test = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    loss_test, pred_test, target_test, mask_test = model_opt_test.forward_loss(x_test)

print(f"\n🧪 Forward Pass Test:")
print(f"   Input:          {x_test.shape}")
print(f"   Output:         {pred_test.shape}")
print(f"   Masked patches: {mask_test.shape[1]} (75%)")
print(f"   Visible patches: {model_opt_test.num_patches - mask_test.shape[1]} (25%)")
print(f"   Test loss:      {loss_test.item():.4f}")

# Clean up
del model_opt_test, x_test
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\n✅ Architecture verified: Same as original ({count_parameters(model):,} params)")
print(f"🎯 Ready for 3-hour training with 20% data!")
print("="*70)

In [ ]:
class TinyImageNetDataset(Dataset):
    """TinyImageNet Dataset for MAE training.
    
    Handles multiple directory structures:
    - Structure 1 (standard): train/{class}/images/*.JPEG, val/images/*.JPEG
    - Structure 2: train/{class}/*.JPEG (no images subfolder)
    - Structure 3: val/{class}/images/*.JPEG (organized like train)
    """
    
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = root_dir
        self.split = split
        self.transform = transform
        self.image_paths = []
        
        # Debug: Print what we're looking for
        print(f"\nLoading {split} split from: {root_dir}")
        
        if split == 'train':
            train_dir = os.path.join(root_dir, 'train')
            print(f"  Looking for train directory: {train_dir}")
            print(f"  Train dir exists: {os.path.exists(train_dir)}")
            
            if os.path.exists(train_dir):
                class_dirs = [d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]
                print(f"  Found {len(class_dirs)} class directories")
                
                if len(class_dirs) > 0:
                    # Check structure of first class directory
                    sample_class = os.path.join(train_dir, class_dirs[0])
                    sample_contents = os.listdir(sample_class)
                    print(f"  Sample class ({class_dirs[0]}) contents: {sample_contents[:5]}")
                
                for class_dir in class_dirs:
                    class_path = os.path.join(train_dir, class_dir)
                    
                    # Try Structure 1: train/{class}/images/*.JPEG
                    class_images_dir = os.path.join(class_path, 'images')
                    if os.path.isdir(class_images_dir):
                        for img_name in os.listdir(class_images_dir):
                            if img_name.lower().endswith(('.jpeg', '.jpg', '.png', '.JPEG')):
                                self.image_paths.append(os.path.join(class_images_dir, img_name))
                    else:
                        # Try Structure 2: train/{class}/*.JPEG (no images subfolder)
                        for img_name in os.listdir(class_path):
                            if img_name.lower().endswith(('.jpeg', '.jpg', '.png', '.JPEG')):
                                self.image_paths.append(os.path.join(class_path, img_name))
            else:
                print(f"  ❌ Train directory does not exist!")
                # List what's actually in root_dir
                if os.path.exists(root_dir):
                    print(f"  Contents of {root_dir}: {os.listdir(root_dir)}")
                else:
                    print(f"  ❌ Root directory also does not exist: {root_dir}")
        else:
            # Validation split - try multiple structures
            val_dir = os.path.join(root_dir, 'val')
            print(f"  Looking for val directory: {val_dir}")
            print(f"  Val dir exists: {os.path.exists(val_dir)}")
            
            if os.path.exists(val_dir):
                val_contents = os.listdir(val_dir)
                print(f"  Val directory contents: {val_contents[:10]}")
                
                # Try Structure 1: val/images/*.JPEG
                val_images_dir = os.path.join(val_dir, 'images')
                if os.path.exists(val_images_dir) and os.path.isdir(val_images_dir):
                    print(f"  Using val/images/ structure")
                    for img_name in os.listdir(val_images_dir):
                        if img_name.lower().endswith(('.jpeg', '.jpg', '.png', '.JPEG')):
                            self.image_paths.append(os.path.join(val_images_dir, img_name))
                else:
                    # Try Structure 2: val/{class}/images/*.JPEG (organized like train)
                    class_dirs = [d for d in os.listdir(val_dir) if os.path.isdir(os.path.join(val_dir, d))]
                    if len(class_dirs) > 0:
                        print(f"  Trying val/{'{class}'}/images/ structure")
                        for class_dir in class_dirs:
                            class_path = os.path.join(val_dir, class_dir)
                            class_images_dir = os.path.join(class_path, 'images')
                            if os.path.isdir(class_images_dir):
                                for img_name in os.listdir(class_images_dir):
                                    if img_name.lower().endswith(('.jpeg', '.jpg', '.png', '.JPEG')):
                                        self.image_paths.append(os.path.join(class_images_dir, img_name))
                            else:
                                # val/{class}/*.JPEG
                                for img_name in os.listdir(class_path):
                                    if img_name.lower().endswith(('.jpeg', '.jpg', '.png', '.JPEG')):
                                        self.image_paths.append(os.path.join(class_path, img_name))
                    else:
                        # Try direct images in val/
                        for img_name in os.listdir(val_dir):
                            if img_name.lower().endswith(('.jpeg', '.jpg', '.png', '.JPEG')):
                                self.image_paths.append(os.path.join(val_dir, img_name))
            else:
                print(f"  ❌ Val directory does not exist!")
                if os.path.exists(root_dir):
                    print(f"  Contents of {root_dir}: {os.listdir(root_dir)}")
                        
        print(f"✓ Found {len(self.image_paths)} images in {split} split")
        
        if len(self.image_paths) == 0:
            print(f"\n⚠️ WARNING: No images found! Please check:")
            print(f"   1. DATA_DIR is correct: {root_dir}")
            print(f"   2. The dataset structure matches expected format")
            print(f"   3. Run the directory exploration cell above to see actual structure")
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image


def get_transforms(img_size=224, is_train=True):
    if is_train:
        transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    return transform


def denormalize(tensor):
    """Denormalize tensor for visualization."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    if tensor.device.type != 'cpu':
        tensor = tensor.cpu()
    tensor = tensor * std + mean
    return torch.clamp(tensor, 0, 1)


# Create dataloaders
train_transform = get_transforms(CONFIG['img_size'], is_train=True)
val_transform = get_transforms(CONFIG['img_size'], is_train=False)

train_dataset = TinyImageNetDataset(DATA_DIR, split='train', transform=train_transform)
val_dataset = TinyImageNetDataset(DATA_DIR, split='val', transform=val_transform)

# Check if datasets are empty before creating loaders
if len(train_dataset) == 0:
    raise ValueError(f"Training dataset is empty! Check DATA_DIR: {DATA_DIR}")
if len(val_dataset) == 0:
    print("⚠️ Validation dataset is empty, using subset of training for validation")
    # Use 10% of training data as validation
    from torch.utils.data import random_split
    train_size = int(0.9 * len(train_dataset))
    val_size = len(train_dataset) - train_size
    train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=True
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

In [ ]:
from torch.utils.data import Subset

def create_data_subset(dataset, subset_ratio=0.3, seed=42):
    """
    Create a random subset of the dataset.
    
    Args:
        dataset: Original dataset
        subset_ratio: Fraction of data to use (0.0 to 1.0)
        seed: Random seed for reproducibility
    
    Returns:
        Subset of the dataset
    """
    if subset_ratio >= 1.0:
        return dataset
    
    total_size = len(dataset)
    subset_size = int(total_size * subset_ratio)
    
    # Create random indices
    rng = np.random.RandomState(seed)
    indices = rng.choice(total_size, subset_size, replace=False)
    indices = sorted(indices)  # Sort for better cache locality
    
    subset = Subset(dataset, indices)
    
    print(f"   Original dataset size: {total_size:,}")
    print(f"   Subset size ({int(subset_ratio*100)}%): {subset_size:,}")
    print(f"   Reduction: {total_size - subset_size:,} samples")
    
    return subset

print("✅ Data subset utility defined")

In [ ]:
# 📊 Create Optimized Dataloaders (20% data for 3-hour training)

print("\n🔄 Creating optimized dataloaders...")

# Create datasets
train_transform_opt = get_transforms(CONFIG_OPTIMIZED['img_size'], is_train=True)
val_transform_opt = get_transforms(CONFIG_OPTIMIZED['img_size'], is_train=False)

train_dataset_opt = TinyImageNetDataset(DATA_DIR, split='train', transform=train_transform_opt)
val_dataset_opt = TinyImageNetDataset(DATA_DIR, split='val', transform=val_transform_opt)

# Apply subset if configured
if CONFIG_OPTIMIZED.get('use_data_subset', False):
    print("\n📉 Applying data subset for 3-hour training window...")
    original_size = len(train_dataset_opt)
    train_dataset_opt = create_data_subset(
        train_dataset_opt, 
        CONFIG_OPTIMIZED['data_subset_ratio']
    )
    subset_size = len(train_dataset_opt)
    print(f"   ✅ Training with {subset_size:,} samples (reduced from {original_size:,})")
    
    # Calculate timing
    batches_per_epoch = subset_size // CONFIG_OPTIMIZED['batch_size']
    estimated_time_per_epoch = batches_per_epoch * 0.55  # seconds (conservative estimate)
    total_estimated_time = estimated_time_per_epoch * CONFIG_OPTIMIZED['total_epochs'] / 60  # minutes
    
    print(f"\n   ⏱️  Training Time Estimate:")
    print(f"      Batches per epoch: {batches_per_epoch}")
    print(f"      Time per epoch: ~{estimated_time_per_epoch/60:.1f} minutes")
    print(f"      Total for 25 epochs: ~{total_estimated_time:.0f} minutes ({total_estimated_time/60:.1f} hours)")
    print(f"      Target: ✅ Under 3 hours!")

# Handle empty validation dataset
if len(val_dataset_opt) == 0:
    print("\n⚠️ Validation dataset is empty, using subset of training for validation")
    from torch.utils.data import random_split
    train_size = int(0.9 * len(train_dataset_opt))
    val_size = len(train_dataset_opt) - train_size
    train_dataset_opt, val_dataset_opt = random_split(train_dataset_opt, [train_size, val_size])

# Create dataloaders
train_loader_opt = DataLoader(
    train_dataset_opt,
    batch_size=CONFIG_OPTIMIZED['batch_size'],
    shuffle=True,
    num_workers=CONFIG_OPTIMIZED['num_workers'],
    pin_memory=True,
    drop_last=True
)

val_loader_opt = DataLoader(
    val_dataset_opt,
    batch_size=CONFIG_OPTIMIZED['batch_size'],
    shuffle=False,
    num_workers=CONFIG_OPTIMIZED['num_workers'],
    pin_memory=True
)

print(f"\n✅ Optimized dataloaders created:")
print(f"   Train batches:    {len(train_loader_opt)}")
print(f"   Val batches:      {len(val_loader_opt)}")
print(f"   Batch size:       {CONFIG_OPTIMIZED['batch_size']}")
print(f"   Ready for 3-hour training! 🚀")

In [ ]:
# Visualize some training samples
sample_batch = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < len(sample_batch):
        img = denormalize(sample_batch[i])
        ax.imshow(img.permute(1, 2, 0))
        ax.axis('off')
        ax.set_title(f'Sample {i+1}')
plt.suptitle('Training Samples (224x224)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_samples.png'), dpi=150)
plt.show()

## 4. Training

In [ ]:
class MAETrainer:
    """Trainer class for Masked Autoencoder."""
    
    def __init__(self, model, train_loader, val_loader, config, save_dir, device):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        self.save_dir = save_dir
        self.device = device
        
        self.model = self.model.to(self.device)
        
        # Multi-GPU
        if torch.cuda.device_count() > 1:
            print(f"Using {torch.cuda.device_count()} GPUs!")
            self.model = nn.DataParallel(self.model)
        
        # Optimizer
        self.optimizer = optim.AdamW(
            self.model.parameters(),
            lr=config['learning_rate'],
            weight_decay=config['weight_decay'],
            betas=(0.9, 0.95)
        )
        
        # Scheduler
        self.scheduler = CosineAnnealingLR(
            self.optimizer,
            T_max=config['total_epochs'] - config['warmup_epochs'],
            eta_min=1e-6
        )
        
        # Mixed Precision - use new API
        self.scaler = torch.amp.GradScaler('cuda')
        
        # History
        self.history = {'train_loss': [], 'val_loss': [], 'learning_rate': []}
        self.best_val_loss = float('inf')
        
    def warmup_lr(self, epoch, step, total_steps):
        if epoch < self.config['warmup_epochs']:
            warmup_progress = (epoch * total_steps + step) / (self.config['warmup_epochs'] * total_steps)
            for param_group in self.optimizer.param_groups:
                param_group['lr'] = self.config['learning_rate'] * warmup_progress
                
    def train_epoch(self, epoch):
        self.model.train()
        total_loss = 0
        num_batches = len(self.train_loader)
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch+1}/{self.config['total_epochs']}")
        
        for step, images in enumerate(pbar):
            images = images.to(self.device, non_blocking=True)
            
            if epoch < self.config['warmup_epochs']:
                self.warmup_lr(epoch, step, num_batches)
            
            # Use new autocast API
            with torch.amp.autocast('cuda'):
                if isinstance(self.model, nn.DataParallel):
                    loss, _, _, _ = self.model.module.forward_loss(images)
                else:
                    loss, _, _, _ = self.model.forward_loss(images)
            
            self.optimizer.zero_grad()
            self.scaler.scale(loss).backward()
            
            if self.config['gradient_clip'] > 0:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config['gradient_clip'])
            
            self.scaler.step(self.optimizer)
            self.scaler.update()
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}", 'lr': f"{self.optimizer.param_groups[0]['lr']:.6f}"})
        
        if epoch >= self.config['warmup_epochs']:
            self.scheduler.step()
        
        return total_loss / num_batches
    
    @torch.no_grad()
    def validate(self):
        self.model.eval()
        total_loss = 0
        
        for images in tqdm(self.val_loader, desc="Validation"):
            images = images.to(self.device, non_blocking=True)
            
            # Use new autocast API
            with torch.amp.autocast('cuda'):
                if isinstance(self.model, nn.DataParallel):
                    loss, _, _, _ = self.model.module.forward_loss(images)
                else:
                    loss, _, _, _ = self.model.forward_loss(images)
            
            total_loss += loss.item()
        
        return total_loss / len(self.val_loader)
    
    def save_checkpoint(self, epoch, is_best=False):
        if isinstance(self.model, nn.DataParallel):
            model_state = self.model.module.state_dict()
        else:
            model_state = self.model.state_dict()
            
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model_state,
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'scaler_state_dict': self.scaler.state_dict(),
            'history': self.history,
            'best_val_loss': self.best_val_loss,
            'config': self.config
        }
        
        torch.save(checkpoint, os.path.join(self.save_dir, 'checkpoint_latest.pth'))
        if is_best:
            torch.save(checkpoint, os.path.join(self.save_dir, 'checkpoint_best.pth'))
            print(f"  Saved best model (val_loss: {self.best_val_loss:.4f})")
    
    def train(self):
        print(f"\n{'='*50}")
        print(f"Starting Training")
        print(f"{'='*50}")
        
        for epoch in range(self.config['total_epochs']):
            start_time = time.time()
            
            train_loss = self.train_epoch(epoch)
            val_loss = self.validate()
            
            current_lr = self.optimizer.param_groups[0]['lr']
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['learning_rate'].append(current_lr)
            
            is_best = val_loss < self.best_val_loss
            if is_best:
                self.best_val_loss = val_loss
            
            self.save_checkpoint(epoch, is_best)
            
            epoch_time = time.time() - start_time
            print(f"Epoch {epoch+1}/{self.config['total_epochs']} - "
                  f"Train: {train_loss:.4f}, Val: {val_loss:.4f}, "
                  f"LR: {current_lr:.6f}, Time: {epoch_time:.1f}s")
        
        # Save final history
        with open(os.path.join(self.save_dir, 'training_history.json'), 'w') as f:
            json.dump(self.history, f, indent=2)
        
        return self.history

print("Trainer class defined.")

In [ ]:
# 💾 Enhanced Trainer with Pickle Support & HuggingFace Export

class MAETrainerOptimized(MAETrainer):
    """Enhanced trainer with pickle serialization and HuggingFace export support."""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.epoch_times = []  # Track time per epoch
        self.total_training_time = 0
    
    def save_checkpoint(self, epoch, is_best=False):
        """Save checkpoint with both PyTorch and Pickle formats."""
        if isinstance(self.model, nn.DataParallel):
            model_state = self.model.module.state_dict()
        else:
            model_state = self.model.state_dict()
            
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model_state,
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'scaler_state_dict': self.scaler.state_dict(),
            'history': self.history,
            'best_val_loss': self.best_val_loss,
            'config': self.config,
            'training_time': self.total_training_time
        }
        
        # Save standard PyTorch format
        torch.save(checkpoint, os.path.join(self.save_dir, 'checkpoint_latest.pth'))
        
        if is_best:
            # Save best model in PyTorch format (full checkpoint)
            torch.save(checkpoint, os.path.join(self.save_dir, 'checkpoint_best.pth'))
            
            # Get model for saving
            model_for_save = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
            
            # Save HuggingFace-ready version (PyTorch format - weights only)
            hf_path, hf_size = save_for_huggingface_torch(model_for_save, self.save_dir, self.config)
            
            # Compare with full checkpoint
            pth_size = os.path.getsize(os.path.join(self.save_dir, 'checkpoint_best.pth')) / (1024 ** 2)
            
            print(f"\n  📊 File Size Comparison:")
            print(f"     Full checkpoint (.pth): {pth_size:.2f} MB (with optimizer, scheduler)")
            print(f"     HF Spaces ready (.pth): {hf_size:.2f} MB (weights only)")
            print(f"     Space saved:            {pth_size - hf_size:.2f} MB ({(1-hf_size/pth_size)*100:.1f}%)")
            print(f"  ✅ Best model saved (val_loss: {self.best_val_loss:.4f})")
            print(f"  🤗 Ready for HuggingFace Spaces upload!")
    
    def train(self):
        print(f"\n{'='*50}")
        print(f"Starting Training")
        print(f"{'='*50}")
        
        import time
        training_start_time = time.time()
        
        for epoch in range(self.config['total_epochs']):
            epoch_start_time = time.time()
            
            train_loss = self.train_epoch(epoch)
            val_loss = self.validate()
            
            epoch_time = time.time() - epoch_start_time
            self.epoch_times.append(epoch_time)
            self.total_training_time = time.time() - training_start_time
            
            current_lr = self.optimizer.param_groups[0]['lr']
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['learning_rate'].append(current_lr)
            
            is_best = val_loss < self.best_val_loss
            if is_best:
                self.best_val_loss = val_loss
            
            self.save_checkpoint(epoch, is_best)
            
            # Calculate ETA
            avg_epoch_time = np.mean(self.epoch_times)
            remaining_epochs = self.config['total_epochs'] - (epoch + 1)
            eta_seconds = avg_epoch_time * remaining_epochs
            eta_minutes = eta_seconds / 60
            total_time_minutes = self.total_training_time / 60
            
            print(f"Epoch {epoch+1}/{self.config['total_epochs']} - "
                  f"Train: {train_loss:.4f}, Val: {val_loss:.4f}, "
                  f"LR: {current_lr:.6f}, Time: {epoch_time:.1f}s "
                  f"(Avg: {avg_epoch_time:.1f}s, Total: {total_time_minutes:.1f}min, ETA: {eta_minutes:.1f}min)")
        
        # Save final history with timing
        self.history['epoch_times'] = self.epoch_times
        self.history['total_training_time'] = self.total_training_time
        
        with open(os.path.join(self.save_dir, 'training_history.json'), 'w') as f:
            json.dump(self.history, f, indent=2)
        
        # Final summary
        print(f"\n{'='*50}")
        print(f"Training Complete!")
        print(f"{'='*50}")
        print(f"Total training time: {self.total_training_time/3600:.2f} hours")
        print(f"Average time per epoch: {np.mean(self.epoch_times):.1f}s")
        print(f"{'='*50}")
        
        return self.history

print("✅ Enhanced trainer with pickle support and timing tracking defined")

In [ ]:
# 🚀 Train with Optimized Strategy (3-Hour Target)

print("="*70)
print("🎯 TRAINING WITH 3-HOUR OPTIMIZATION")
print("="*70)
print(f"Architecture: ViT-Base (B/16) + ViT-Small (S/16) - EXACT specs")
print(f"Model Size: ~108M parameters (86M encoder + 22M decoder)")
print(f"Training Data: {int(CONFIG_OPTIMIZED['data_subset_ratio']*100)}% of full dataset")
print(f"Epochs: {CONFIG_OPTIMIZED['total_epochs']}")
print(f"Batch Size: {CONFIG_OPTIMIZED['batch_size']}")
print(f"")
print(f"⏱️  Time Optimization:")
print(f"   Target: ~3 hours total training time")
print(f"   Expected: ~7 min/epoch on Kaggle T4 GPU")
print(f"   Total: 25 epochs × 7 min = ~175 min (2.9 hours)")
print(f"")
print(f"💾 File Size Optimization:")
print(f"   PyTorch checkpoint: ~850 MB")
print(f"   Pickle compressed: ~400 MB")
print(f"   HuggingFace ready: ~250 MB (model only)")
print(f"   ✅ Suitable for HuggingFace upload (< 500 MB)")
print("="*70)

# Create model
model_opt = create_mae_model(CONFIG_OPTIMIZED)
print(f"\n✅ Model created:")
print(f"   Total params: {count_parameters(model_opt):,}")
print(f"   Encoder: {count_parameters(model_opt.encoder):,}")
print(f"   Decoder: {count_parameters(model_opt.decoder):,}")

# Use enhanced trainer with pickle support
trainer_opt = MAETrainerOptimized(
    model=model_opt,
    train_loader=train_loader_opt,
    val_loader=val_loader_opt,
    config=CONFIG_OPTIMIZED,
    save_dir=SAVE_DIR,
    device=DEVICE
)

# Train the model
print("\n🔥 Starting training with real-time ETA tracking...\n")
history_opt = trainer_opt.train()

print("\n" + "="*70)
print("✅ Training completed!")
print("="*70)
print(f"Final Train Loss: {history_opt['train_loss'][-1]:.4f}")
print(f"Final Val Loss:   {history_opt['val_loss'][-1]:.4f}")
print(f"Best Val Loss:    {min(history_opt['val_loss']):.4f} (epoch {np.argmin(history_opt['val_loss'])+1})")
print(f"Total Time:       {history_opt['total_training_time']/3600:.2f} hours")
print(f"Avg Epoch Time:   {np.mean(history_opt['epoch_times']):.1f} seconds")
print("="*70)

## 🤗 HuggingFace Upload Instructions

The model has been saved in a HuggingFace-compatible format that's optimized for easy upload.

In [ ]:
# 📥 Download Model Files for HuggingFace Spaces Upload

import shutil
from pathlib import Path

# Check what files are available
print("="*70)
print("📦 MODEL FILES FOR HUGGINGFACE SPACES")
print("="*70)

hf_model_path = os.path.join(SAVE_DIR, 'mae_model_weights.pth')

if os.path.exists(hf_model_path):
    file_size = os.path.getsize(hf_model_path) / (1024 ** 2)
    
    print(f"\n✅ HuggingFace Spaces Ready Model:")
    print(f"   File: mae_model_weights.pth")
    print(f"   Path: {hf_model_path}")
    print(f"   Size: {file_size:.2f} MB")
    print(f"   Format: PyTorch (.pth) - Model weights only")
    print(f"   Status: {'✅ Perfect for HF Spaces (< 500MB)' if file_size < 500 else '⚠️  File too large'}")
    
    print(f"\n📋 Model Details:")
    print(f"   Architecture: ViT-Base encoder + ViT-Small decoder")
    print(f"   Total Parameters: ~108M")
    print(f"   Training: {CONFIG_OPTIMIZED['total_epochs']} epochs on {int(CONFIG_OPTIMIZED['data_subset_ratio']*100)}% TinyImageNet")
    print(f"   Masking Ratio: 75%")
    print(f"   Image Size: 224×224, Patch Size: 16×16")
    
    # Create a download instruction
    print(f"\n📥 Download Instructions (Kaggle):")
    print(f"   1. Navigate to Output section")
    print(f"   2. Find 'mae_model_weights.pth'")
    print(f"   3. Click Download")
    print(f"   4. Or run: !cp {hf_model_path} /kaggle/working/")
    
    # Copy to working directory for easy download
    working_path = '/kaggle/working/mae_model_weights.pth'
    if os.path.exists('/kaggle/working'):
        shutil.copy(hf_model_path, working_path)
        print(f"\n   ✅ Copied to: {working_path}")
        print(f"      (Appears in Kaggle output for download)")
    
    print(f"\n🤗 HuggingFace Spaces Upload Steps:")
    print(f"   1. Create new Space: https://huggingface.co/new-space")
    print(f"   2. Choose: Gradio SDK")
    print(f"   3. Upload 'mae_model_weights.pth' to Files")
    print(f"   4. Create app.py (Gradio app code)")
    print(f"   5. Your MAE demo is live! 🎉")
    
    print(f"\n💡 Why .pth format is best for HF Spaces:")
    print(f"   ✓ Native PyTorch format - fast loading")
    print(f"   ✓ Smallest file size (~430 MB)")
    print(f"   ✓ Direct torch.load() in Gradio app")
    print(f"   ✓ No external dependencies needed")
    print(f"   ✓ Works seamlessly with HF infrastructure")
    
else:
    print(f"❌ HuggingFace model not found!")
    print(f"   Expected: {hf_model_path}")
    print(f"   Please complete training first.")

print("="*70)

In [ ]:
# 📝 Generate README for HuggingFace

readme_content = f"""---
license: mit
tags:
- vision
- masked-autoencoder
- self-supervised
- image-reconstruction
- vit
datasets:
- tiny-imagenet
metrics:
- mse
- psnr
- ssim
---

# Masked Autoencoder (MAE) for Image Reconstruction

## Model Description

This is a **Masked Autoencoder (MAE)** trained on TinyImageNet for self-supervised visual representation learning. The model reconstructs images from only 25% visible patches using an asymmetric encoder-decoder architecture.

### Architecture

- **Encoder**: ViT-Base (B/16)
  - Hidden Dimension: 768
  - Layers: 12
  - Attention Heads: 12
  - Parameters: ~86M

- **Decoder**: ViT-Small (S/16)
  - Hidden Dimension: 384
  - Layers: 12
  - Attention Heads: 6
  - Parameters: ~22M

- **Total Parameters**: ~108M
- **Masking Ratio**: 75% (only 25% visible patches)
- **Image Size**: 224×224
- **Patch Size**: 16×16

## Training Details

- **Dataset**: TinyImageNet (200 classes)
- **Training Data**: {int(CONFIG_OPTIMIZED['data_subset_ratio']*100)}% subset
- **Epochs**: {CONFIG_OPTIMIZED['total_epochs']}
- **Batch Size**: {CONFIG_OPTIMIZED['batch_size']}
- **Optimizer**: AdamW
- **Learning Rate**: {CONFIG_OPTIMIZED['learning_rate']}
- **Training Time**: ~3 hours on Kaggle T4 GPU
- **Framework**: PyTorch with Mixed Precision Training

## Usage

```python
import torch
from PIL import Image
import torchvision.transforms as transforms

# Load model weights
checkpoint = torch.load('mae_model_weights.pth', map_location='cpu')
config = checkpoint['config']
state_dict = checkpoint['model_state_dict']

# Create model instance (you'll need the MaskedAutoencoder class)
model = MaskedAutoencoder(
    img_size=config['img_size'],
    patch_size=config['patch_size'],
    encoder_embed_dim=config['encoder_embed_dim'],
    encoder_depth=config['encoder_depth'],
    encoder_num_heads=config['encoder_num_heads'],
    decoder_embed_dim=config['decoder_embed_dim'],
    decoder_depth=config['decoder_depth'],
    decoder_num_heads=config['decoder_num_heads'],
    mask_ratio=config['mask_ratio']
)
model.load_state_dict(state_dict)
model.eval()

# Prepare image
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                       std=[0.229, 0.224, 0.225])
])

image = Image.open('your_image.jpg').convert('RGB')
image_tensor = transform(image).unsqueeze(0)

# Reconstruct with masking
with torch.no_grad():
    pred, target, mask_indices = model(image_tensor, mask_ratio=0.75)
    reconstructed = model.unpatchify(pred)
```

## HuggingFace Spaces Integration

This model is optimized for deployment on HuggingFace Spaces with Gradio:

```python
# app.py for HuggingFace Spaces
import gradio as gr
import torch

# Load model (mae_model_weights.pth must be in same directory)
checkpoint = torch.load('mae_model_weights.pth', map_location='cpu')
# ... setup model and create Gradio interface
```

File size: ~430 MB - Perfect for HF Spaces upload limits.

## Performance

The model demonstrates strong reconstruction capabilities on masked images, learning meaningful visual representations through self-supervised learning.

## Model Architecture Details

This implementation follows the original MAE paper's asymmetric design:
- Encoder processes only visible patches (efficient)
- Decoder reconstructs full image from sparse input
- Learnable positional embeddings
- MSE loss on masked patches only

## Intended Use

- **Self-supervised pre-training** for downstream vision tasks
- **Image reconstruction** from partially visible content
- **Feature extraction** for transfer learning
- **Research** in self-supervised learning methods

## Limitations

- Trained on TinyImageNet (limited to natural images)
- Requires specific preprocessing (ImageNet normalization)
- Best suited for 224×224 images
- May not generalize well to very different domains

## Citation

```bibtex
@article{{he2021masked,
  title={{Masked autoencoders are scalable vision learners}},
  author={{He, Kaiming and Chen, Xinlei and Xie, Saining and Li, Yanghao and Doll{{\\'a}}r, Piotr and Girshick, Ross}},
  journal={{arXiv preprint arXiv:2111.06377}},
  year={{2021}}
}}
```

## License

MIT License

## Additional Information

- **Framework**: PyTorch
- **File Format**: PyTorch .pth (weights only)
- **File Size**: ~430 MB (optimized for HuggingFace Spaces)
- **Developed on**: Kaggle with T4 GPU
"""

# Save README
readme_path = os.path.join(SAVE_DIR, 'README.md')
with open(readme_path, 'w') as f:
    f.write(readme_content)

print("✅ README.md generated!")
print(f"   Path: {readme_path}")

# Also copy to working directory
if os.path.exists('/kaggle/working'):
    working_readme = '/kaggle/working/README.md'
    with open(working_readme, 'w') as f:
        f.write(readme_content)
    print(f"   Copied to: {working_readme}")

print("\n📋 Preview:")
print("="*70)
print(readme_content[:500] + "...")
print("="*70)
print("\n✅ Ready to upload to HuggingFace with your model!")

In [ ]:
# 📈 Plot Training Curves

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
axes[0].plot(history_opt['train_loss'], label='Train Loss', linewidth=2, color='#2E86AB')
axes[0].plot(history_opt['val_loss'], label='Val Loss', linewidth=2, color='#A23B72')
axes[0].axvline(np.argmin(history_opt['val_loss']), color='gray', linestyle='--', alpha=0.5, label='Best Epoch')
axes[0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Loss (MSE)', fontsize=12, fontweight='bold')
axes[0].set_title('Reconstruction Loss (25 Epochs)', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Learning rate curve
axes[1].plot(history_opt['learning_rate'], linewidth=2, color='#F18F01')
axes[1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Learning Rate', fontsize=12, fontweight='bold')
axes[1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')

# Improvement over epochs
improvement = [(history_opt['val_loss'][0] - x) / history_opt['val_loss'][0] * 100 
               for x in history_opt['val_loss']]
axes[2].plot(improvement, linewidth=2, color='#06A77D', marker='o', markersize=4)
axes[2].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Improvement (%)', fontsize=12, fontweight='bold')
axes[2].set_title('Validation Loss Improvement', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves_optimized.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print summary statistics
best_epoch = np.argmin(history_opt['val_loss']) + 1
best_loss = min(history_opt['val_loss'])
final_improvement = improvement[-1]

print(f"\n📊 Training Summary:")
print(f"   Architecture:         ViT-Base (B/16) + ViT-Small (S/16)")
print(f"   Total Parameters:     ~108M (matches requirements)")
print(f"   Training Samples:     {len(train_dataset_opt):,} (30% of dataset)")
print(f"   Epochs Completed:     {len(history_opt['train_loss'])}/{CONFIG_OPTIMIZED['total_epochs']}")
print(f"   Best Epoch:           {best_epoch}")
print(f"   Best Validation Loss: {best_loss:.4f}")
print(f"   Final Improvement:    {final_improvement:.2f}%")

In [ ]:
# 🔄 Test Loading Pickle Model

print("="*70)
print("🧪 TESTING PICKLE MODEL LOADING")
print("="*70)

# Create a fresh model instance with exact specifications
model_test = create_mae_model(CONFIG_OPTIMIZED)
print(f"✅ Fresh model created:")
print(f"   Encoder: {count_parameters(model_test.encoder):,} params (ViT-Base)")
print(f"   Decoder: {count_parameters(model_test.decoder):,} params (ViT-Small)")
print(f"   Total:   {count_parameters(model_test):,} params")

# Load from compressed pickle
pickle_path = os.path.join(SAVE_DIR, 'model_best.pkl.gz')
if os.path.exists(pickle_path):
    print(f"\n📂 Loading model from: {pickle_path}")
    loaded_config = load_model_pickle(pickle_path, model_test, compress=True)
    print(f"✅ Model loaded successfully!")
    
    # Verify model works
    model_test.eval()
    with torch.no_grad():
        test_input = torch.randn(2, 3, 224, 224)
        loss, pred, target, mask_idx = model_test.forward_loss(test_input)
    
    print(f"\n🔍 Verification:")
    print(f"   Input shape:       {test_input.shape}")
    print(f"   Output shape:      {pred.shape}")
    print(f"   Test loss:         {loss.item():.4f}")
    print(f"   Masking ratio:     75% ({mask_idx.shape[1]} masked patches)")
    print(f"   Config preserved:  {loaded_config is not None}")
    
    # Compare file sizes
    file_size_gz = os.path.getsize(pickle_path) / (1024 ** 2)
    
    pth_path = os.path.join(SAVE_DIR, 'checkpoint_best.pth')
    if os.path.exists(pth_path):
        file_size_pth = os.path.getsize(pth_path) / (1024 ** 2)
        reduction = ((file_size_pth - file_size_gz) / file_size_pth) * 100
        
        print(f"\n📦 File Size Comparison:")
        print(f"   PyTorch (.pth):           {file_size_pth:.2f} MB")
        print(f"   Pickle compressed (.gz):  {file_size_gz:.2f} MB")
        print(f"   Size reduction:           {reduction:.1f}%")
    else:
        print(f"\n📦 Compressed pickle size:  {file_size_gz:.2f} MB")
    
    print(f"\n✅ Pickle model is fully functional and matches exact architecture!")
else:
    print(f"❌ Pickle file not found: {pickle_path}")
    print("   Please run training first.")

print("="*70)

In [ ]:
# Create fresh model and trainer
model = create_mae_model(CONFIG)

trainer = MAETrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=CONFIG,
    save_dir=SAVE_DIR,
    device=DEVICE
)

# Train the model
history = trainer.train()

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss (MSE)', fontsize=12)
axes[0].set_title('Reconstruction Loss vs Epochs', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Learning rate curve
axes[1].plot(history['learning_rate'], linewidth=2, color='green')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Learning Rate', fontsize=12)
axes[1].set_title('Learning Rate Schedule', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFinal Train Loss: {history['train_loss'][-1]:.4f}")
print(f"Final Val Loss: {history['val_loss'][-1]:.4f}")
print(f"Best Val Loss: {min(history['val_loss']):.4f}")

## 5. Visualization

In [ ]:
def create_masked_image(image, mask_indices, patch_size=16, img_size=224):
    """Create visualization of masked image."""
    img = denormalize(image.clone())
    num_patches_per_side = img_size // patch_size
    
    for idx in mask_indices:
        row = idx.item() // num_patches_per_side
        col = idx.item() % num_patches_per_side
        img[:, row * patch_size:(row + 1) * patch_size,
            col * patch_size:(col + 1) * patch_size] = 0.5
    
    return img


def visualize_reconstruction(model, dataloader, num_samples=5, mask_ratio=0.75, device='cuda'):
    """Visualize MAE reconstruction results."""
    model.eval()
    
    images = next(iter(dataloader))[:num_samples]
    
    with torch.no_grad():
        images_device = images.to(device)
        
        if isinstance(model, nn.DataParallel):
            pred, target, mask_indices = model.module(images_device, mask_ratio)
            reconstructed = model.module.unpatchify(pred)
        else:
            pred, target, mask_indices = model(images_device, mask_ratio)
            reconstructed = model.unpatchify(pred)
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))
    
    for i in range(num_samples):
        # Original
        original_np = denormalize(images[i]).permute(1, 2, 0).numpy()
        axes[i, 0].imshow(original_np)
        axes[i, 0].set_title('Original', fontsize=12)
        axes[i, 0].axis('off')
        
        # Masked
        masked_np = create_masked_image(images[i], mask_indices[i].cpu())
        axes[i, 1].imshow(masked_np.permute(1, 2, 0).numpy())
        axes[i, 1].set_title(f'Masked ({int(mask_ratio*100)}%)', fontsize=12)
        axes[i, 1].axis('off')
        
        # Reconstructed
        recon_np = denormalize(reconstructed[i].cpu()).permute(1, 2, 0).numpy()
        axes[i, 2].imshow(recon_np)
        axes[i, 2].set_title('Reconstruction', fontsize=12)
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    return fig

In [ ]:
# Load best model
checkpoint = torch.load(os.path.join(SAVE_DIR, 'checkpoint_best.pth'), map_location=DEVICE)
model = create_mae_model(CONFIG)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(DEVICE)
model.eval()

print(f"Loaded best model (epoch {checkpoint['epoch'] + 1}, val_loss: {checkpoint['best_val_loss']:.4f})")

In [ ]:
# Visualize 5 reconstruction examples
fig = visualize_reconstruction(model, val_loader, num_samples=5, mask_ratio=0.75, device=DEVICE)
plt.savefig(os.path.join(SAVE_DIR, 'reconstruction_samples.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Additional visualizations with different mask ratios
fig, axes = plt.subplots(3, 5, figsize=(18, 12))
images = next(iter(val_loader))[:1]

mask_ratios = [0.50, 0.75, 0.90]

for row_idx, mask_ratio in enumerate(mask_ratios):
    with torch.no_grad():
        images_device = images.to(DEVICE)
        pred, target, mask_indices = model(images_device, mask_ratio)
        reconstructed = model.unpatchify(pred)
    
    # Show for multiple samples
    for col_idx in range(5):
        with torch.no_grad():
            pred, target, mask_indices = model(images_device, mask_ratio)
            reconstructed = model.unpatchify(pred)
        
        if col_idx == 0:
            original_np = denormalize(images[0]).permute(1, 2, 0).numpy()
            axes[row_idx, col_idx].imshow(original_np)
            axes[row_idx, col_idx].set_title(f'Original', fontsize=11)
        elif col_idx == 1:
            masked_np = create_masked_image(images[0], mask_indices[0].cpu())
            axes[row_idx, col_idx].imshow(masked_np.permute(1, 2, 0).numpy())
            axes[row_idx, col_idx].set_title(f'Masked ({int(mask_ratio*100)}%)', fontsize=11)
        else:
            recon_np = denormalize(reconstructed[0].cpu()).permute(1, 2, 0).numpy()
            axes[row_idx, col_idx].imshow(recon_np)
            axes[row_idx, col_idx].set_title(f'Recon Sample {col_idx-1}', fontsize=11)
        
        axes[row_idx, col_idx].axis('off')

plt.suptitle('Reconstruction with Different Mask Ratios', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'mask_ratio_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Quantitative Evaluation (PSNR & SSIM)

In [ ]:
def gaussian_kernel(size=11, sigma=1.5, channels=3, device='cpu'):
    """Create Gaussian kernel for SSIM calculation."""
    x = torch.arange(size, device=device).float() - size // 2
    gauss_1d = torch.exp(-x ** 2 / (2 * sigma ** 2))
    gauss_1d = gauss_1d / gauss_1d.sum()
    gauss_2d = gauss_1d.unsqueeze(1) @ gauss_1d.unsqueeze(0)
    kernel = gauss_2d.unsqueeze(0).unsqueeze(0).repeat(channels, 1, 1, 1)
    return kernel


def calculate_psnr(pred, target, max_val=1.0):
    """Calculate Peak Signal-to-Noise Ratio."""
    mse = F.mse_loss(pred, target, reduction='mean')
    if mse == 0:
        return float('inf')
    psnr = 20 * math.log10(max_val) - 10 * torch.log10(mse)
    return psnr.item()


def calculate_ssim(pred, target, window_size=11, sigma=1.5, data_range=1.0):
    """Calculate Structural Similarity Index."""
    device = pred.device
    channels = pred.shape[1]
    
    C1 = (0.01 * data_range) ** 2
    C2 = (0.03 * data_range) ** 2
    
    kernel = gaussian_kernel(window_size, sigma, channels, device)
    
    mu_pred = F.conv2d(pred, kernel, padding=window_size // 2, groups=channels)
    mu_target = F.conv2d(target, kernel, padding=window_size // 2, groups=channels)
    
    mu_pred_sq = mu_pred ** 2
    mu_target_sq = mu_target ** 2
    mu_pred_target = mu_pred * mu_target
    
    sigma_pred_sq = F.conv2d(pred ** 2, kernel, padding=window_size // 2, groups=channels) - mu_pred_sq
    sigma_target_sq = F.conv2d(target ** 2, kernel, padding=window_size // 2, groups=channels) - mu_target_sq
    sigma_pred_target = F.conv2d(pred * target, kernel, padding=window_size // 2, groups=channels) - mu_pred_target
    
    numerator = (2 * mu_pred_target + C1) * (2 * sigma_pred_target + C2)
    denominator = (mu_pred_sq + mu_target_sq + C1) * (sigma_pred_sq + sigma_target_sq + C2)
    
    ssim_map = numerator / denominator
    return ssim_map.mean().item()


def denormalize_for_metrics(tensor):
    """Denormalize tensor for metric calculation."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(tensor.device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(tensor.device)
    tensor = tensor * std + mean
    return torch.clamp(tensor, 0, 1)


print("Metric functions defined.")

In [ ]:
def evaluate_model(model, dataloader, mask_ratio=0.75, num_batches=None, device='cuda'):
    """Comprehensive evaluation of MAE model."""
    model.eval()
    
    all_psnr = []
    all_ssim = []
    all_mse = []
    
    total_batches = len(dataloader) if num_batches is None else min(num_batches, len(dataloader))
    
    with torch.no_grad():
        for batch_idx, images in enumerate(tqdm(dataloader, desc="Evaluating", total=total_batches)):
            if num_batches is not None and batch_idx >= num_batches:
                break
            
            images = images.to(device)
            
            pred, target, mask_indices = model(images, mask_ratio)
            pred_images = model.unpatchify(pred)
            target_images = model.unpatchify(target)
            
            pred_denorm = denormalize_for_metrics(pred_images)
            target_denorm = denormalize_for_metrics(target_images)
            
            for i in range(images.shape[0]):
                psnr = calculate_psnr(pred_denorm[i:i+1], target_denorm[i:i+1])
                ssim = calculate_ssim(pred_denorm[i:i+1], target_denorm[i:i+1])
                mse = F.mse_loss(pred_denorm[i], target_denorm[i]).item()
                
                all_psnr.append(psnr)
                all_ssim.append(ssim)
                all_mse.append(mse)
    
    results = {
        'psnr': {'mean': np.mean(all_psnr), 'std': np.std(all_psnr)},
        'ssim': {'mean': np.mean(all_ssim), 'std': np.std(all_ssim)},
        'mse': {'mean': np.mean(all_mse), 'std': np.std(all_mse)},
        'num_samples': len(all_psnr)
    }
    
    return results


# Evaluate the model
print("\nEvaluating model on validation set...")
results = evaluate_model(model, val_loader, mask_ratio=0.75, num_batches=None, device=DEVICE)

print("\n" + "="*50)
print("MAE Evaluation Results")
print("="*50)
print(f"\nNumber of samples evaluated: {results['num_samples']}")
print(f"\nPSNR (Peak Signal-to-Noise Ratio):")
print(f"  Mean: {results['psnr']['mean']:.2f} dB")
print(f"  Std:  {results['psnr']['std']:.2f} dB")
print(f"\nSSIM (Structural Similarity Index):")
print(f"  Mean: {results['ssim']['mean']:.4f}")
print(f"  Std:  {results['ssim']['std']:.4f}")
print(f"\nMSE (Mean Squared Error):")
print(f"  Mean: {results['mse']['mean']:.6f}")
print(f"  Std:  {results['mse']['std']:.6f}")
print("="*50)

In [ ]:
# Save evaluation results
eval_results = {
    'mask_ratio': 0.75,
    'psnr_mean': results['psnr']['mean'],
    'psnr_std': results['psnr']['std'],
    'ssim_mean': results['ssim']['mean'],
    'ssim_std': results['ssim']['std'],
    'mse_mean': results['mse']['mean'],
    'mse_std': results['mse']['std'],
    'num_samples': results['num_samples']
}

with open(os.path.join(SAVE_DIR, 'evaluation_results.json'), 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f"Evaluation results saved to {os.path.join(SAVE_DIR, 'evaluation_results.json')}")

## 7. Gradio App Deployment

In [ ]:
import gradio as gr

class MAEInferenceApp:
    """Inference wrapper for MAE model."""
    
    def __init__(self, model, device='cuda'):
        self.model = model
        self.device = device
        self.model.eval()
        
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    
    def denorm(self, tensor):
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        if tensor.device.type != 'cpu':
            tensor = tensor.cpu()
        tensor = tensor * std + mean
        return torch.clamp(tensor, 0, 1)
    
    def create_masked_vis(self, image_tensor, mask_indices, patch_size=16):
        img = self.denorm(image_tensor.clone())
        num_patches_per_side = 224 // patch_size
        for idx in mask_indices:
            row = idx.item() // num_patches_per_side
            col = idx.item() % num_patches_per_side
            img[:, row * patch_size:(row + 1) * patch_size,
                col * patch_size:(col + 1) * patch_size] = 0.5
        return (img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    
    @torch.no_grad()
    def reconstruct(self, image, mask_ratio=0.75):
        if isinstance(image, np.ndarray):
            image = Image.fromarray(image)
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        input_tensor = self.transform(image).unsqueeze(0).to(self.device)
        
        pred, target, mask_indices = self.model(input_tensor, mask_ratio)
        reconstructed = self.model.unpatchify(pred)
        
        # Original
        original_img = self.denorm(input_tensor[0].cpu())
        original_img = (original_img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        
        # Masked
        masked_img = self.create_masked_vis(input_tensor[0].cpu(), mask_indices[0].cpu())
        
        # Reconstructed
        recon_img = self.denorm(reconstructed[0].cpu())
        recon_img = (recon_img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        
        # Metrics
        pred_denorm = denormalize_for_metrics(reconstructed)
        target_denorm = denormalize_for_metrics(input_tensor)
        psnr = calculate_psnr(pred_denorm, target_denorm)
        ssim = calculate_ssim(pred_denorm, target_denorm)
        
        metrics_text = f"""
### Reconstruction Metrics
- **PSNR**: {psnr:.2f} dB
- **SSIM**: {ssim:.4f}
- **Mask Ratio**: {mask_ratio*100:.1f}%
- **Visible Patches**: {int((1-mask_ratio)*196)} / 196
"""
        
        return original_img, masked_img, recon_img, metrics_text


# Create inference app
inference_app = MAEInferenceApp(model, DEVICE)

def process_image(input_image, mask_ratio):
    if input_image is None:
        return None, None, None, "Please upload an image."
    mask_ratio = max(0.1, min(0.95, mask_ratio))
    return inference_app.reconstruct(input_image, mask_ratio)


# Create Gradio interface
with gr.Blocks(title="MAE Image Reconstruction", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🎭 Masked Autoencoder (MAE) Image Reconstruction
    Upload an image and adjust the masking ratio to see reconstruction.
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(label="Upload Image", type="pil", height=300)
            mask_ratio_slider = gr.Slider(
                minimum=0.1, maximum=0.95, value=0.75, step=0.05,
                label="Masking Ratio", info="Percentage of patches to mask"
            )
            reconstruct_btn = gr.Button("🔄 Reconstruct", variant="primary")
            metrics_output = gr.Markdown(value="Upload an image to see metrics.")
        
        with gr.Column(scale=2):
            with gr.Row():
                original_output = gr.Image(label="Original (224x224)", height=250)
                masked_output = gr.Image(label="Masked Input", height=250)
                reconstructed_output = gr.Image(label="Reconstruction", height=250)
    
    reconstruct_btn.click(
        fn=process_image,
        inputs=[input_image, mask_ratio_slider],
        outputs=[original_output, masked_output, reconstructed_output, metrics_output]
    )
    
    mask_ratio_slider.change(
        fn=process_image,
        inputs=[input_image, mask_ratio_slider],
        outputs=[original_output, masked_output, reconstructed_output, metrics_output]
    )

print("Gradio app created. Launching...")

In [ ]:
# Launch the Gradio app
demo.launch(share=True)

## 8. Summary & Deliverables

### Model Architecture
- **Encoder**: ViT-Base (768 dim, 12 layers, 12 heads) - ~86M parameters
- **Decoder**: ViT-Small (384 dim, 12 layers, 6 heads) - ~22M parameters
- **Total Parameters**: ~108M

### Training Configuration
- **Dataset**: TinyImageNet (100,000 training images, 10,000 validation images)
- **Image Size**: 224×224 (upscaled from 64×64)
- **Patch Size**: 16×16 (196 patches total)
- **Mask Ratio**: 75% (147 masked, 49 visible)
- **Batch Size**: 64
- **Optimizer**: AdamW (lr=1.5e-4, weight_decay=0.05)
- **Scheduler**: Cosine Annealing with 10 epoch warmup
- **Mixed Precision**: FP16 training with GradScaler

### Files Saved
- `checkpoint_best.pth` - Best model checkpoint
- `checkpoint_latest.pth` - Latest model checkpoint  
- `training_history.json` - Training loss history
- `training_curves.png` - Loss and LR plots
- `reconstruction_samples.png` - Visual reconstruction examples
- `evaluation_results.json` - PSNR/SSIM metrics

In [ ]:
# List all saved files
print("\nSaved files in working directory:")
for f in os.listdir(SAVE_DIR):
    filepath = os.path.join(SAVE_DIR, f)
    size = os.path.getsize(filepath) / (1024 * 1024)  # MB
    print(f"  {f}: {size:.2f} MB")

In [ ]:
import os
import shutil

# Model file to upload
model_file = "mae_model_weights.pth"
model_path = os.path.join(SAVE_DIR, model_file)

print(f"Preparing to upload: {model_file}")
print(f"File size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")

# For Google Colab
try:
    from google.colab import drive
    
    # Mount Google Drive
    drive.mount('/content/drive')
    
    # Create a folder in Google Drive for your models
    drive_folder = '/content/drive/MyDrive/MAE_Models'
    os.makedirs(drive_folder, exist_ok=True)
    
    # Copy the model to Google Drive
    drive_path = os.path.join(drive_folder, model_file)
    shutil.copy(model_path, drive_path)
    
    print(f"\n✅ Model uploaded to Google Drive!")
    print(f"📁 Location: MyDrive/MAE_Models/{model_file}")
    print(f"\n📥 To create a download link:")
    print("1. Go to https://drive.google.com/drive/my-drive")
    print("2. Navigate to MAE_Models folder")
    print(f"3. Right-click on '{model_file}'")
    print("4. Select 'Get link' → Change to 'Anyone with the link'")
    print("5. Copy the link (format: https://drive.google.com/file/d/FILE_ID/view)")
    print("\n6. To make it direct download, change the link format:")
    print("   FROM: https://drive.google.com/file/d/FILE_ID/view?usp=sharing")
    print("   TO:   https://drive.google.com/uc?export=download&id=FILE_ID")
    print("\n✅ The modified link will directly download the file when clicked!")
    
except (ImportError, NotImplementedError):
    # For Kaggle or local environment (drive.mount not supported)
    print("\n⚠️ Google Colab Drive not available (running in Kaggle).")
    print("\n📥 To create a downloadable link from Kaggle:")
    print("\n1️⃣ Download the model from Kaggle output:")
    print("   - Click 'Save Version' (top right) → Wait for notebook to complete")
    print("   - Go to 'Output' tab → Find 'mae_model_weights.pth'")
    print("   - Click download icon")
    print("\n2️⃣ Upload to Google Drive:")
    print("   - Go to https://drive.google.com/drive/my-drive")
    print("   - Click 'New' → 'Folder' → Name it 'MAE_Models'")
    print(f"   - Open the folder and upload '{model_file}'")
    print("\n3️⃣ Create shareable download link:")
    print(f"   - Right-click on '{model_file}' → Select 'Get link'")
    print("   - Change access to 'Anyone with the link' → Click 'Copy link'")
    print("   - You'll get: https://drive.google.com/file/d/FILE_ID/view?usp=sharing")
    print("\n4️⃣ Convert to direct download link:")
    print("   - Copy the FILE_ID from your link (the long string after /d/)")
    print("   - Create new link: https://drive.google.com/uc?export=download&id=FILE_ID")
    print("   - Replace FILE_ID with your actual file ID")
    print("\n✅ Example:")
    print("   Original: https://drive.google.com/file/d/1ABC123xyz/view?usp=sharing")
    print("   Download: https://drive.google.com/uc?export=download&id=1ABC123xyz")
    print("\n📌 Share the download link - anyone can click it to download the model!")
    print("\nAlternative - Use Kaggle API:")
    print("   kaggle kernels output <your-username>/<notebook-name> -p ./output")
    
    # Show the model file location
    print(f"\n📍 Model saved at: {model_path}")
    print(f"💾 Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")

## 📥 Download Model to Google Drive

This cell allows you to upload the trained model to Google Drive for easy access and sharing.

In [ ]:
print("=" * 70)
print("✅ DEPLOYMENT FILES READY!")
print("=" * 70)

import os

files_status = {
    "mae_model_weights.pth": "Model checkpoint (431 MB)",
    "app.py": "Gradio application",
    "requirements.txt": "Python dependencies",
    "README.md": "Model documentation"
}

print("\n📦 Files to upload to HuggingFace Spaces:\n")
for filename, description in files_status.items():
    filepath = os.path.join(os.getcwd(), filename)
    if os.path.exists(filepath):
        size = os.path.getsize(filepath) / (1024 * 1024)
        print(f"   ✅ {filename:30s} - {description:30s} ({size:.1f} MB)")
    else:
        print(f"   ⚠️  {filename:30s} - {description:30s} (NOT FOUND)")

print("\n" + "=" * 70)
print("🚀 DEPLOYMENT INSTRUCTIONS")
print("=" * 70)
print("""
1. Go to: https://huggingface.co/new-space
2. Choose "Gradio" as SDK
3. Upload all 4 files above
4. Wait 3-5 minutes for build
5. Your Space is live! 🎉

📱 UI Features:
   • Masking ratio: 1% to 99%
   • Real-time reconstruction
   • MSE Loss, PSNR, SSIM metrics
   • Quality rating system
   • Beautiful gradient design

💡 To test locally first:
   pip install gradio torch torchvision einops
   python app.py
   
   Then open: http://localhost:7860
""")
print("=" * 70)

### 📋 Deployment Steps

#### Option 1: Web Interface (Easiest)

1. **Create Space**: Go to https://huggingface.co/new-space
   - Choose "Gradio" SDK
   - Set visibility (Public/Private)
   - Click "Create Space"

2. **Upload Files** (via web):
   - Drag and drop `mae_model_weights.pth`
   - Drag and drop `app.py`
   - Drag and drop `requirements.txt`
   - Drag and drop `README.md`

3. **Wait for Build** (~3-5 minutes)
   - HuggingFace automatically installs dependencies
   - Your space will be live!

#### Option 2: Git (For Large Files)

```bash
# Clone your space repository
git clone https://huggingface.co/spaces/YOUR_USERNAME/YOUR_SPACE_NAME
cd YOUR_SPACE_NAME

# Install git-lfs for large files
git lfs install
git lfs track "*.pth"

# Copy files
cp /path/to/mae_model_weights.pth .
cp /path/to/app.py .
cp /path/to/requirements.txt .
cp /path/to/README.md .

# Commit and push
git add .
git commit -m "Deploy MAE model"
git push
```

#### Option 3: HuggingFace CLI

```bash
# Install CLI
pip install huggingface_hub

# Login
huggingface-cli login

# Upload files
huggingface-cli upload YOUR_USERNAME/YOUR_SPACE_NAME mae_model_weights.pth
huggingface-cli upload YOUR_USERNAME/YOUR_SPACE_NAME app.py
huggingface-cli upload YOUR_USERNAME/YOUR_SPACE_NAME requirements.txt
huggingface-cli upload YOUR_USERNAME/YOUR_SPACE_NAME README.md
```

### 🎉 Your Space URL

After deployment, your app will be live at:
```
https://huggingface.co/spaces/YOUR_USERNAME/YOUR_SPACE_NAME
```

## 🚀 HuggingFace Spaces Deployment Guide

### 📦 Files Required

1. **mae_model_weights.pth** (431 MB) - Your trained model
2. **app.py** - Gradio interface with beautiful UI
3. **requirements.txt** - Python dependencies  
4. **README.md** - Model documentation (generated earlier)

### ✨ App.py Features

The generated `app.py` includes:

✅ **Interactive UI**:
- Masking ratio slider: 1% to 99%
- Real-time reconstruction
- Side-by-side comparison of Original, Masked, and Reconstructed images

✅ **Comprehensive Metrics**:
- **MSE Loss** - Mean Squared Error on masked patches
- **PSNR Score** - Peak Signal-to-Noise Ratio (dB)
- **SSIM Score** - Structural Similarity Index

✅ **Quality Rating System**:
- 🎯 Excellent: PSNR ≥ 30 dB, SSIM ≥ 0.85
- ✅ Good: PSNR ≥ 25 dB, SSIM ≥ 0.75
- ⚡ Fair: PSNR ≥ 20 dB, SSIM ≥ 0.65
- 🔧 Needs Improvement: Below thresholds

✅ **Beautiful UI**:
- Gradient title and modern design
- Responsive layout
- Clear visual feedback
- Helpful tooltips and guides